# 🤖 CATIA V5 Upstage Solar 기반 자동 질문 생성 및 RAG 고격차 검증 파이프라인

## 📌 개요 및 목적
본 노트북은 **Upstage Solar Pro (`solar-pro`)**를 활용하여 **ChromaDB에 저장된 32개 CATIA V5 매뉴얼 문맥**을 읽고, **RAG가 적용되었을 때 대답 성능이 크게 오르는 고난도 질문과 표준 정답(Ground Truth)**을 자동으로 생성합니다.

생성된 질문은 **Qwen 2.5 0.5B (`Qwen/Qwen2.5-0.5B-Instruct` 로컬 PyTorch 모델)**을 통해 **Direct LLM(RAG Off)**과 **RAG LLM(RAG On)** 답변을 유도한 뒤, **Upstage Solar LLM-as-a-Judge**의 1~5점 정밀 검증을 거칩니다.

**최종적으로 `Solar Score(RAG On) > Solar Score(RAG Off)` 조건(RAG 효과가 입증된 질문)을 만족하는 질문만 선별**하여 설정된 목표 개수(사전 설정: 20개)만큼 수집하고, 결과를 `eval/CATIA_RAG_High_Gap_Auto_Questions.csv` 파일로 자동 저장합니다.

--- 
### ⚙️ 주요 설정 변수
- **`TARGET_QUESTION_COUNT = 20`**: 최종 선별할 RAG 고격차 우수 질문 목표 개수
- **질문 생성기 & 평가관**: Upstage Solar LLM (`solar-pro` via Upstage API Key)
- **검증 대상 소형 모델**: Qwen 2.5 0.5B (`Qwen/Qwen2.5-0.5B-Instruct` 100% 로컬 PyTorch)

In [6]:
# 1. 환경 설정 및 모듈 임포트
import os
import sys
import gc
import json
import random
import pandas as pd
import torch
from pathlib import Path
from bert_score import score as bert_score_compute
from IPython.display import HTML, display
from langchain_openai import ChatOpenAI

# 프로젝트 루트 sys.path 추가
PROJECT_ROOT = Path(".").resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT / ".." / "src").exists():
    PROJECT_ROOT = (PROJECT_ROOT / "..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.rag_chain import RAGPipeline
from src.vector_store import build_or_load_vectorstore
from src.config import DATA_DIR, CHROMA_DB_DIR

# Upstage API Key 및 파라미터 설정
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY", "up_UFLAwDwDhV9YXWlUKU6CI3piyvp9q")
os.environ["UPSTAGE_API_KEY"] = UPSTAGE_API_KEY

TARGET_QUESTION_COUNT = 20  # 사전에 설정된 목표 고격차 질문 수 (초기값: 20개)

print(f"[System] PyTorch CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[System] Active GPU: {torch.cuda.get_device_name(0)}")
print(f"[System] Upstage API Key Loaded: {UPSTAGE_API_KEY[:8]}...")
print(f"[System] Target High-Gap Question Count: {TARGET_QUESTION_COUNT}개")


[System] PyTorch CUDA Available: True
[System] Active GPU: NVIDIA GeForce RTX 4070 Laptop GPU
[System] Upstage API Key Loaded: up_UFLAw...
[System] Target High-Gap Question Count: 20개


In [7]:
# 2. VectorStore 로딩 및 PDF 문맥 추출기/Upstage Solar 객체 생성
print("[VectorStore] Loading ChromaDB RAG Vector Store...")
vectorstore = build_or_load_vectorstore()

# ChromaDB 컬렉션 갯수 점검 (0개이면 자동 재구축)
if vectorstore._collection.count() == 0:
    print("[VectorStore Warning] Vector Store is empty (0 docs). Rebuilding from data/*.pdf...")
    vectorstore = build_or_load_vectorstore(force_rebuild=True)
    print(f"[VectorStore] Rebuilt completed. Total docs in collection: {vectorstore._collection.count()}")
else:
    print(f"[VectorStore] Loaded existing collection. Total docs: {vectorstore._collection.count()}")

# PDF 원문 직접 추출 백업 로더
from src.multimodal_loader import load_and_split_multimodal_pdf
pdf_chunks = []
try:
    pdf_chunks = load_and_split_multimodal_pdf()
    print(f"[PDF Loader] Backup PDF Text Chunks Loaded: {len(pdf_chunks)} chunks.")
except Exception as e:
    print(f"[PDF Loader Warning] Backup PDF loader failed: {e}")

solar_llm = ChatOpenAI(
    model="solar-pro",
    openai_api_key=UPSTAGE_API_KEY,
    openai_api_base="https://api.upstage.ai/v1/solar",
    temperature=0.7
)

solar_judge = ChatOpenAI(
    model="solar-pro",
    openai_api_key=UPSTAGE_API_KEY,
    openai_api_base="https://api.upstage.ai/v1/solar",
    temperature=0.0
)

# 고격차(UI/단축키/조작법) 유도용 질문 생성 프롬프트
GENERATE_QUESTION_PROMPT = """당신은 CATIA V5 전문 CAD 출제자입니다.
아래 제공된 CATIA V5 매뉴얼 문서 내용을 바탕으로, 일반 LLM(사전학습 지식만 있는 모델)은 알지 못해 1점(환각/오답)을 받고, RAG 문서 문맥을 참조한 모델은 5점(정답)을 받을 수 있는 '고격차 퀴즈 질문 1개'와 '표준 정답'을 생성하세요.

[출제 지침 - RAG 고격차 유도 핵심]
1. 단순 개념 설명 질문은 지양하고, CATIA V5 전용 키보드 단축키(e.g., F3, Alt+Enter, Ctrl+U, Space), 마우스 드래그 조합(e.g., 중간 휠+오른쪽 클릭 드래그), 특정 툴바 아이콘 명칭(e.g., Snap to Point, Exit Workbench, Swap Visible Space, Smart Move, Red Handle, Multi-View), 대화상자 세부 옵션/탭 이름(e.g., Break and Trim, Multi-Pad limits, Hole types)을 명확하게 묻는 질문을 만드세요.
2. 질문은 단정적이고 정밀해야 합니다.

[매뉴얼 참고 문맥]:
{context}

반드시 아래 JSON 형식으로만 응답하세요:
{{
  "question": "<생성된 질문>",
  "reference_answer": "<표준 정답>",
  "reference_sentence": "<근거 매뉴얼 문장 요약>",
  "question_type": "<ui_shortcut 또는 specific_ui>"
}}"""

SOLAR_EVAL_PROMPT = """당신은 CATIA V5 전문 CAD 평가관(LLM-as-a-Judge)입니다.
전문적인 CATIA V5 CAD 지식에 기반하여, 아래 질문에 대해 AI 모델이 생성한 답변의 정확성을 1점부터 5점까지 점수로 직접 평가해주세요.

[평가 점수 기준]
- 5점 (맞음 / 매우 정확함): 질문에서 요구한 CATIA V5의 정확한 기능명, 키보드 단축키, 마우스 조작법 및 절차가 오류 없이 명확하게 설명됨.
- 4점 (부분 맞음): 핵심 기능명이나 단축키가 대체로 맞으나 설명이 약간 부족하거나 사소한 미흡함이 있음.
- 3점 (보통 / 애매함): 일반적인 CAD 개념 언급은 있으나 질문에서 요청한 핵심 단축키나 명칭이 누락되어 모호함.
- 2점 (틀림 / 불일치): 엉뚱한 용어를 언급하거나 다른 CAD 기능과 혼동하여 오답에 가까움.
- 1점 (전혀 아님 / 환각): 존재하지 않는 단축키/기능을 지어내거나(환각), 무응답 또는 질문과 전혀 상관없는 오답.

[질문]: {question}
[모델 생성 답변]: {generated_answer}

반드시 아래 JSON 형식으로만 응답하세요:
{{
  "score": <1부터 5 사이 정수>,
  "reason": "<한 줄 평가 요약>"
}}"""

print("[Upstage Generator & Evaluator] Initialized successfully.")


[VectorStore] Loading ChromaDB RAG Vector Store...
[VectorStore] Initializing Local Embeddings 'sentence-transformers/all-MiniLM-L6-v2'...
[VectorStore] Loading existing Chroma database from: C:\KDT_14\[11]Transformer\project\team-03-project\vect\chroma_db_multimodal
[VectorStore Warning] Vector Store is empty (0 docs). Rebuilding from data/*.pdf...
[VectorStore] Initializing Local Embeddings 'sentence-transformers/all-MiniLM-L6-v2'...
[VectorStore] Building new Multimodal Chroma database at: C:\KDT_14\[11]Transformer\project\team-03-project\vect\chroma_db_multimodal
[MultimodalLoader] Found 32 PDF manual files in 'C:\KDT_14\[11]Transformer\project\team-03-project\data'. Extracting multimodal text & diagrams...
[MultimodalLoader] Total 2416 pages extracted across 32 PDF files.
[VectorStore] Multimodal vector store successfully created and persisted.
[VectorStore] Rebuilt completed. Total docs in collection: 5278
[MultimodalLoader] Found 32 PDF manual files in 'C:\KDT_14\[11]Transformer

In [8]:
# 3. 기존 질문 불러오기, 중복 방지 및 신규 질문 추가(Append) 추론 루프
MODEL_NAME = "Qwen 2.5 0.5B"
REPO_ID = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"==================================================")
print(f"[Pipeline] Initializing Local Qwen Model: {MODEL_NAME} ({REPO_ID})")
print(f"==================================================")
qwen_pipeline = RAGPipeline(model_name=REPO_ID, mode="local")

csv_save_path = PROJECT_ROOT / "eval" / "CATIA_RAG_High_Gap_Auto_Questions.csv"
txt_save_path = PROJECT_ROOT / "eval" / "CATIA_RAG_High_Gap_Questions_Only.txt"

# 1) 기존 생성된 CSV 파일이 존재하면 불러와 유지 (Append 준비)
valid_high_gap_records = []
existing_questions_set = set()

if csv_save_path.exists():
    try:
        df_old = pd.read_csv(csv_save_path)
        valid_high_gap_records = df_old.to_dict("records")
        for q in df_old["question"].dropna():
            existing_questions_set.add(str(q).strip())
        print(f"[CSV Check] 기존에 누적 저장된 질문 {len(valid_high_gap_records)}개를 성공적으로 불러왔습니다.")
    except Exception as e:
        print(f"[CSV Check Warning] 기존 파일 로딩 중 오류: {e}")

TARGET_NEW_ADDITIONS = 20  # 재실행 시 새로 추가할 신규 고격차 질문 수
initial_count = len(valid_high_gap_records)
target_total_count = initial_count + TARGET_NEW_ADDITIONS

attempt_count = 0

print(f"\n🚀 [Auto QA Append Loop] 기존 {initial_count}개 질문 유지 + 신규 고격차 질문 {TARGET_NEW_ADDITIONS}개 추가 수집 시작 (목표 누적: {target_total_count}개)...\n")

def evaluate_answer_solar(q, ans):
    try:
        prompt = SOLAR_EVAL_PROMPT.format(question=q, generated_answer=ans)
        res = solar_judge.invoke(prompt)
        text = res.content.strip()
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0].strip()
        elif "```" in text:
            text = text.split("```")[1].split("```")[0].strip()
        data = json.loads(text)
        score = int(data.get("score", 1))
        reason = str(data.get("reason", ""))
        return min(max(score, 1), 5), reason
    except Exception:
        return 1, "평가 예외 발생"

def get_random_context_chunk():
    try:
        if vectorstore._collection.count() > 0:
            res = vectorstore._collection.get(limit=300)
            docs = [d for d in res.get("documents", []) if d and len(d) > 80]
            if docs:
                return random.choice(docs)[:1000]
    except Exception:
        pass
    if pdf_chunks:
        c = random.choice(pdf_chunks)
        return c.page_content[:1000]
    return "CATIA V5 Specification Tree toggle shortcut is F3. Compass tool is used for 3D rotation and translation. Update icon shortcut is Ctrl+U."

while len(valid_high_gap_records) < target_total_count and attempt_count < 100:
    attempt_count += 1
    new_added = len(valid_high_gap_records) - initial_count
    print(f"--- [Attempt #{attempt_count}] (신규 추가 현황: {new_added}/{TARGET_NEW_ADDITIONS}개 | 총 누적: {len(valid_high_gap_records)}개) ---")
    
    # 1) 문맥 획득
    chunk_text = get_random_context_chunk()
    
    # 2) Upstage Solar로 RAG 고격차 질문 생성
    try:
        gen_res = solar_llm.invoke(GENERATE_QUESTION_PROMPT.format(context=chunk_text))
        gen_text = gen_res.content.strip()
        if "```json" in gen_text:
            gen_text = gen_text.split("```json")[1].split("```")[0].strip()
        elif "```" in gen_text:
            gen_text = gen_text.split("```")[1].split("```")[0].strip()
        q_data = json.loads(gen_text)
        
        q_text = q_data.get("question", "").strip()
        gt_text = q_data.get("reference_answer", "").strip()
        ref_sentence = q_data.get("reference_sentence", "").strip()
        q_type = q_data.get("question_type", "ui_shortcut")
        
        if not q_text or len(q_text) < 8:
            continue
            
        # ⭐ [중복 방지 검사] 기존에 작성된 질문과 동일하거나 유사하면 스킵
        if q_text in existing_questions_set or any(q_text[:20] in eq for eq in existing_questions_set):
            print(f"  🔄 [Duplicate Skipped] 이미 존재하는 질문이므로 스킵합니다: {q_text[:30]}...")
            continue
    except Exception as e:
        print(f"  ⚠️ Question Generation Error: {e}")
        continue
        
    # 3) Qwen 0.5B 추론 (Direct vs RAG)
    ans_direct = qwen_pipeline.answer_direct(q_text)
    res_rag = qwen_pipeline.answer_rag(q_text)
    ans_rag = res_rag.get("answer", "")
    sources = res_rag.get("source_pages", [])
    source_str = ", ".join(sources) if sources else "매뉴얼 문맥 참조"
    
    # 4) Upstage Solar 평가관 직접 평가
    s_dir, r_dir = evaluate_answer_solar(q_text, ans_direct)
    s_rag, r_rag = evaluate_answer_solar(q_text, ans_rag)
    
    print(f"  Q: {q_text[:45]}...")
    print(f"  Direct Solar Score: {s_dir}/5 | RAG Solar Score: {s_rag}/5 (Score Gap: +{s_rag - s_dir})")
    
    # 5) ⭐ 핵심 필터링 조건: RAG Score가 Direct Score보다 높은 신규 질문만 Append!
    if s_rag > s_dir:
        record_id = f"Q{len(valid_high_gap_records)+1:02d}"
        rec = {
            "id": record_id,
            "question": q_text,
            "reference_answer": gt_text,
            "reference_sentence": ref_sentence,
            "question_type": q_type,
            "direct_answer": ans_direct,
            "rag_answer": ans_rag,
            "source_citation": source_str,
            "solar_direct_score": s_dir,
            "solar_direct_reason": r_dir,
            "solar_rag_score": s_rag,
            "solar_rag_reason": r_rag,
            "solar_score_gap": s_rag - s_dir
        }
        valid_high_gap_records.append(rec)
        existing_questions_set.add(q_text)
        print(f"  ✅ [NEW SELECTION #{len(valid_high_gap_records)}] Appended High-Gap Question (Gap: +{s_rag - s_dir}점)!\n")
    else:
        print(f"  ❌ Discarded (RAG Score was not strictly higher than Direct Score)\n")

print(f"🎉 [Auto QA Append Finished] 총 {len(valid_high_gap_records)}개의 고격차 질문 수집 완료 (이번 회차에 {len(valid_high_gap_records) - initial_count}개 추가됨)!")


[Pipeline] Initializing Local Qwen Model: Qwen 2.5 0.5B (Qwen/Qwen2.5-0.5B-Instruct)
[LLMFactory] Initializing LLM 'Qwen/Qwen2.5-0.5B-Instruct' in mode='local'...
[LLMFactory Local] Loading model 'Qwen/Qwen2.5-0.5B-Instruct' on device 'cuda'...


Device set to use cuda:0


[VectorStore] Initializing Local Embeddings 'sentence-transformers/all-MiniLM-L6-v2'...
[VectorStore] Loading existing Chroma database from: C:\KDT_14\[11]Transformer\project\team-03-project\vect\chroma_db_multimodal

🚀 [Auto QA Loop] Generating and Filtering RAG High-Gap Questions (Target: 20개)...

--- [Attempt #1] (Collected High-Gap Questions: 0/20) ---


c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-p

  Q: CATIA V5에서 Groove 생성 시 'Break and Trim' 옵션을 활...
  Direct Solar Score: 2/5 | RAG Solar Score: 2/5 (Score Gap: +0)
  ❌ Discarded (RAG Score was not strictly higher than Direct Score)

--- [Attempt #2] (Collected High-Gap Questions: 0/20) ---


...

  Q: CATIA V5에서 'Parents/Children' 도구를 실행하기 위해 반드시...
  Direct Solar Score: 1/5 | RAG Solar Score: 2/5 (Score Gap: +1)
  ✅ [SELECTION #18] Accepted High-Gap Question (Gap: +1점)!

--- [Attempt #78] (Collected High-Gap Questions: 18/20) ---


c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-p

  Q: CATIA V5에서 'Red Handle' 기능을 활성화하기 위해 반드시 클릭해야...
  Direct Solar Score: 2/5 | RAG Solar Score: 2/5 (Score Gap: +0)
  ❌ Discarded (RAG Score was not strictly higher than Direct Score)

--- [Attempt #79] (Collected High-Gap Questions: 18/20) ---


c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-p

  Q: CATIA V5에서 툴바에 숨겨진 추가 도구를 확장/축소할 때 사용하는 정확한 마...
  Direct Solar Score: 2/5 | RAG Solar Score: 2/5 (Score Gap: +0)
  ❌ Discarded (RAG Score was not strictly higher than Direct Score)

--- [Attempt #80] (Collected High-Gap Questions: 18/20) ---
  ⚠️ Question Generation Error: Extra data: line 8 column 1 (char 759)
🎉 [Auto QA Loop Finished] Successfully collected 18 High-Gap Questions!


## 👁️ 4. 생성 문장 1:1 Side-by-Side 대조 시각화 표 (Upstage Solar 점수 포함)
Upstage Solar 평가 점수에서 **RAG LLM이 Direct LLM보다 더 우수한 대답을 도출한 필터링된 질문들**에 대한 표준 정답 및 답변 1:1 대조 표입니다.

In [9]:
# 4. Side-by-Side 대조 표 생성 및 HTML 출력
score_labels = {
    5: ("맞음 (5/5)", "#15803d", "#dcfce7"),
    4: ("부분 맞음 (4/5)", "#0369a1", "#e0f2fe"),
    3: ("보통 (3/5)", "#b45309", "#fef3c7"),
    2: ("틀림 (2/5)", "#c2410c", "#ffedd5"),
    1: ("전혀 아님/환각 (1/5)", "#b91c1c", "#fee2e2")
}

css_style = """
<style>
    .eval-table {
        width: 100%;
        border-collapse: collapse;
        font-family: 'Segoe UI', Malgun Gothic, sans-serif;
        font-size: 13px;
        margin-top: 10px;
    }
    .eval-table th {
        background-color: #0f172a;
        color: #ffffff;
        text-align: center;
        padding: 10px;
        border: 1px solid #475569;
        font-weight: 600;
    }
    .eval-table td {
        padding: 10px 12px;
        border: 1px solid #cbd5e1;
        vertical-align: top;
        line-height: 1.5;
        word-break: break-word;
    }
    .col-gt { background-color: #f0fdf4; color: #166534; font-weight: 600; }
    .col-direct { background-color: #fafafa; color: #1e293b; padding: 8px; border-radius: 4px; border: 1px solid #e2e8f0; }
    .col-rag { background-color: #f0f9ff; color: #0369a1; padding: 8px; border-radius: 4px; border: 1px solid #bae6fd; font-weight: 500; }
    .badge { display: inline-block; padding: 3px 8px; border-radius: 4px; font-weight: bold; font-size: 11px; margin-bottom: 6px; }
    .badge-solar { display: inline-block; padding: 3px 8px; border-radius: 4px; font-weight: bold; font-size: 11px; margin-bottom: 6px; border: 1px solid; }
    .badge-direct { background-color: #fda4af; color: #881337; }
    .badge-rag { background-color: #93c5fd; color: #1e3a8a; }
</style>
"""

html_table = css_style + f"""
<div style="overflow-x: auto; border: 1px solid #cbd5e1; border-radius: 6px;">
    <table class="eval-table">
        <thead>
            <tr>
                <th style="width: 50px;">ID</th>
                <th style="width: 200px;">질문 (Question)</th>
                <th style="width: 240px;">표준 정답 (Ground Truth)</th>
                <th style="width: 330px;">{MODEL_NAME} Direct LLM (RAG Off)</th>
                <th style="width: 330px;">{MODEL_NAME} RAG LLM (RAG On)</th>
            </tr>
        </thead>
        <tbody>
"""

for rec in valid_high_gap_records:
    qid = rec["id"]
    q = rec["question"]
    gt = rec["reference_answer"]
    ans_dir = rec["direct_answer"]
    ans_rag = rec["rag_answer"]
    src = rec["source_citation"]
    
    s_dir = rec["solar_direct_score"]
    r_dir = rec["solar_direct_reason"]
    lbl_dir, col_dir, bg_dir = score_labels.get(s_dir, ("평가전", "#000", "#fff"))
    
    s_rag = rec["solar_rag_score"]
    r_rag = rec["solar_rag_reason"]
    lbl_rag, col_rag, bg_rag = score_labels.get(s_rag, ("평가전", "#000", "#fff"))
    
    html_table += f"""
            <tr>
                <td style="text-align: center; font-weight: bold;">{qid}</td>
                <td><b>{q}</b></td>
                <td class="col-gt">{gt}</td>
                <td>
                    <span class="badge badge-direct">[RAG Off / Direct LLM]</span><br/>
                    <span class="badge-solar" style="background-color: {bg_dir}; color: {col_dir}; border-color: {col_dir};">⭐ Upstage Solar: {lbl_dir}</span><br/>
                    <small style="color: #475569;">💬 {r_dir}</small>
                    <div class="col-direct" style="margin-top: 6px;">{ans_dir}</div>
                </td>
                <td>
                    <span class="badge badge-rag">[RAG On / 매뉴얼 참조]</span><br/>
                    <span class="badge-solar" style="background-color: {bg_rag}; color: {col_rag}; border-color: {col_rag};">⭐ Upstage Solar: {lbl_rag}</span><br/>
                    <small style="color: #475569;">💬 {r_rag}</small>
                    <div class="col-rag" style="margin-top: 6px;">{ans_rag}</div><br/>
                    <small style="color: #475569;">📍 <b>참조 근거:</b> {src}</small>
                </td>
            </tr>
    """

html_table += """
        </tbody>
    </table>
</div>
"""

display(HTML(html_table))


ID,질문 (Question),표준 정답 (Ground Truth),Qwen 2.5 0.5B Direct LLM (RAG Off),Qwen 2.5 0.5B RAG LLM (RAG On)
Q01,CATIA V5에서 복사된 요소를 시스템 클립보드에 저장하는 기능은 어떤 툴 아이콘을 통해 실행할 수 있나요?,Copy,"[RAG Off / Direct LLM] ⭐ Upstage Solar: 틀림 (2/5) 💬 CATIA V5의 실제 시스템 클립보드 연동 기능(예: 'Copy to System Clipboard' 아이콘)을 언급하지 않고, 일반적인 클립보드 개념을 혼동하여 설명함. 핵심 기능명과 절차가 부정확함. CATIA V5에서 복사된 요소를 시스템 클립보드에 저장하는 방법은 다음과 같습니다: 1. **클립보드 열기**: 먼저, 원래의 CAD 파일을 클립보드로 열어야 합니다. 2. **복사된 요소 선택**: 클립보드에서 원래의 CAD 파일을 선택합니다. 이 과정에서 복사된 요소가 포함되어 있을 것입니다. 3. **복사 버튼 클릭**: 클립보드에서 복사 버튼을 클릭하여 복사된 요소를 클립보드에 복사합니다. 4. **저장**: 복사된 요소를 클립보드에 저장하려면, 클립보드에서 ""저장"" 또는 ""저장하기"" 버튼을 클릭합니다. 이 버튼은 클립보드에서 복사된 요소를 저장할 때 사용됩니다. 이러한 단계들을 따르면, 복사된 요소는 시스템 클립보드에 저장될 수 있습니다.","[RAG On / 매뉴얼 참조] ⭐ Upstage Solar: 보통 (3/5) 💬 CATIA V5에서 시스템 클립보드 저장 기능은 'Copy' 툴이 아닌 'Copy to System Clipboard' 아이콘으로 실행되며, 핵심 기능명이 누락되어 모호함. CATIA V5에서 복사된 요소를 시스템 클립보드에 저장하는 기능은 `Copy` 툴 아이콘을 사용할 수 있습니다. 📍 참조 근거: 1주차 (Basic).pdf (P.12), 1주차 (Basic).pdf (P.14), 1주차 (Basic).pdf (P.23), EDU_CAT_EN_V5F_FB_V5R19.pdf (P.1)"
Q02,"CATIA V5에서 Parameterization Analysis 도구를 사용하여 설계 문서의 모든 엔티티 유형을 목록화할 때, 분석 결과를 필터링하거나 특정 엔티티(예: 스케치, 패드, 홀)만을 선택적으로 표시/숨기기 위해 반드시 사용해야 하는 대화상자의 탭 이름은 무엇인가요?","Parameterization Analysis 대화상자에서 'Break and Trim' 탭은 엔티티 유형을 필터링하거나 선택적으로 표시/숨기는 기능을 제공하지 않습니다. 정확한 탭 이름은 매뉴얼에 명시되지 않았으나, 일반적으로 'Type' 또는 'Filter' 탭에서 해당 작업을 수행합니다. (매뉴얼 기준 정답: 'Break and Trim'은 해당 기능과 무관함)","[RAG Off / Direct LLM] ⭐ Upstage Solar: 전혀 아님/환각 (1/5) 💬 평가 예외 발생 CATIA V5에서 Parameterization Analysis 도구를 사용하여 설계 문서의 모든 엔티티 유형을 목록화하는 데 필요한 대화 상자 이름은 다음과 같습니다: 1. **Parameterization Analysis** (설계 문서의 모든 엔티티 유형을 목록화) 2. **Entity Types** (설계 문서의 모든 엔티티 유형) 3. **Entity List** (설계 문서의 모든 엔티티 리스트) 이 세 가지 항목 모두는 필수적인 정보를 제공하며, 각각의 항목은 분석 결과를 필터링하거나 특정 엔티티만 표시/숨기는 기능을 제공합니다.","[RAG On / 매뉴얼 참조] ⭐ Upstage Solar: 틀림 (2/5) 💬 CATIA V5의 Parameterization Analysis 도구에서 필터링/표시/숨기기 기능은 'Filter' 탭에서 수행되며, 답변에 언급된 다른 탭들은 존재하지 않거나 혼동된 정보입니다. CATIA V5에서 Parameterization Analysis 도구를 사용하여 설계 문서의 모든 엔티티 유형을 목록화하고 필터링하거나 특정 엔티티만 선택적으로 표시/숨기기 위해서는 다음과 같은 대화 상자의 탭 이름이 필요합니다: 1. Parameterization Analysis (설계 문서의 모든 엔티티 유형 목록화) 2. Filter (필터링) 3. Select (선택) 4. Show/Hide (표시/숨기기) 이렇게 하면 Parameterization Analysis 도구와 필터링과 선택을 동시에 사용할 수 있습니다. 📍 참조 근거: 1주차 (Basic).pdf (P.12), 1주차 (Basic).pdf (P.13), 1주차 (Basic).pdf (P.14), 1주차 (Basic).pdf (P.23)"
Q03,CATIA V5에서 사용자 정의 뷰를 생성하고 저장된 뷰로 즉시 전환하는 정확한 메뉴 경로는 무엇인가요?,사용자 정의 뷰 생성 시 [View > Named Views]에서 Add 버튼을 클릭한 후 이름을 입력하고 Apply를 선택합니다. 저장된 뷰로 복귀 시에는 동일한 메뉴 경로(View > Named Views)에서 해당 뷰를 선택합니다.,"[RAG Off / Direct LLM] ⭐ Upstage Solar: 틀림 (2/5) 💬 핵심 기능명('뷰' 대신 '뷔페' 오타) 및 메뉴 경로 오류(실제 경로는 View > Views...), 저장/전환 절차 부정확하여 오답에 가까움 CATIA V5에서 사용자 정의 뷔페를 생성하고 저장된 뷔페로 즉시 전환하는 정확한 메뉴 경로는 다음과 같습니다: 1. **사용자 정의 뷔페 생성**: - `File > New > User-Defined View` 또는 `View > New > User-Defined View` 메뉴를 선택합니다. 2. **사용자 정의 뷔페 저장**: - `File > Save As` 메뉴를 사용하여 뷔페를 저장할 수 있습니다. 이 메뉴에서는 ""Save as"" 버튼을 클릭하여 뷔페를 저장할 수 있습니다. 3. **사용자 정의 뷔페 재생성**: - `File > Open` 메뉴를 사용하여 뷔페를 다시 열면, 그 뷔페를 사용자 정의 뷔페로 재생성할 수 있습니다. 이렇게 하면 사용자 정의 뷔페를 생성하고 저장 후, 즉시 사용자 정의 뷔페로 전환할 수 있습니다.","[RAG On / 매뉴얼 참조] ⭐ Upstage Solar: 보통 (3/5) 💬 핵심 기능인 'Named Views' 대신 'View Wizard'를 언급하여 정확하지 않으며, 실제 메뉴 경로와 단축키 누락으로 모호함 CATIA V5에서 사용자 정의 뷰를 생성하고 저장된 뷰로 즉시 전환하는 정확한 메뉴 경로는 다음과 같습니다: 1. **View Wizard** -> **New View** - 이 메뉴는 사용자 정의 뷰를 생성하고 저장된 뷰로 즉시 전환할 수 있는 기본적인 방법입니다. 2. **View Wizard** -> **Save Current View as New View** - 이 메뉴는 현재 사용 중인 뷰를 저장하고 새로운 뷰로 전환할 수 있는 기능입니다. 3. **View Wizard** -> **Open Existing View** - 이 메뉴는 이미 존재하는 뷰를 열고 새로운 뷰로 전환할 수 있는 기능입니다. 4. **View Wizard** -> **Close Current View** - 이 메뉴는 현재 사용 중인 뷰를 닫아 새로운 뷰로 전환할 수 있는 기능입니다. 이러한 메뉴를 통해 사용자 정의 뷰를 생성하고 저장된 뷔로 즉시 전환할 수 있습니다. 📍 참조 근거: 9주차(Assembly Design).pdf (P.1), CATIA V5 Lectures.pdf (P.3), EDU_CAT_EN_V5F_FB_V5R19.pdf (P.272), EDU_CAT_EN_V5F_FB_V5R19.pdf (P.274)"
Q04,"CATIA V5에서 'Reference Elements (Extended)' 기능으로 Sketch 없이 삼각형을 생성할 때, 두 Point의 On Plane 기준 H 값과 V 값을 각각 100과 80으로 지정하려면 어떤 대화상자 탭/옵션과 키보드 단축키를 조합해

## 📊 5. 정량 점수 비교 및 RAG 고격차 질문 CSV 저장
Solar Score 평가 결과 RAG에서 더 높은 점수를 획득한 고격차 질문 리스트를 정리하고 `eval/CATIA_RAG_High_Gap_Auto_Questions.csv` 파일로 새로 저장합니다.

In [10]:
# 5. BERTScore 정량 계산, 요약 표 생성 및 CSV / TXT 자동 갱신 저장
df_results = pd.DataFrame(valid_high_gap_records)

if not df_results.empty:
    direct_cands = df_results["direct_answer"].astype(str).tolist()
    rag_cands = df_results["rag_answer"].astype(str).tolist()
    refs = df_results["reference_answer"].astype(str).tolist()
    
    P_dir, R_dir, F1_dir = bert_score_compute(cands=direct_cands, refs=refs, lang="ko", verbose=False)
    P_rag, R_rag, F1_rag = bert_score_compute(cands=rag_cands, refs=refs, lang="ko", verbose=False)
    
    df_results["direct_f1"] = [round(f, 4) for f in F1_dir.tolist()]
    df_results["rag_f1"] = [round(f, 4) for f in F1_rag.tolist()]
    df_results["bert_f1_gap"] = [round(r - d, 4) for d, r in zip(F1_dir.tolist(), F1_rag.tolist())]
    
    avg_solar_dir = df_results["solar_direct_score"].mean()
    avg_solar_rag = df_results["solar_rag_score"].mean()
    solar_gap_avg = avg_solar_rag - avg_solar_dir
    
    avg_bert_dir = float(F1_dir.mean())
    avg_bert_rag = float(F1_rag.mean())
    bert_gap_avg = avg_bert_rag - avg_bert_dir
    
    # 요약 표
    df_summary = pd.DataFrame([{
        "Model": MODEL_NAME,
        "Total Accumulated Questions": len(df_results),
        "Direct LLM Solar Avg (1~5점)": f"{avg_solar_dir:.2f} / 5.0",
        "RAG LLM Solar Avg (1~5점)": f"{avg_solar_rag:.2f} / 5.0",
        "Solar Score Improvement (+Δ점수)": f"+{solar_gap_avg:.2f}점",
        "Direct LLM BERT F1": f"{avg_bert_dir:.4f}",
        "RAG LLM BERT F1": f"{avg_bert_rag:.4f}",
        "BERT F1 Improvement (+ΔF1)": f"+{bert_gap_avg:.4f}"
    }])
    
    print("=========================================================================================")
    print(f"    {MODEL_NAME} Upstage Solar Auto-Generated High-Gap QA Performance Summary         ")
    print("=========================================================================================")
    display(df_summary)
    print("\n🔹 [누적 수집된 RAG 고격차 질문 세부 점수 표]")
    display(df_results[["id", "question", "solar_direct_score", "solar_rag_score", "solar_score_gap", "direct_f1", "rag_f1", "bert_f1_gap"]].tail(25))
    
    # 1) CSV 갱신 저장 (Append 누적 반영)
    csv_save_path = PROJECT_ROOT / "eval" / "CATIA_RAG_High_Gap_Auto_Questions.csv"
    df_results.to_csv(csv_save_path, index=False, encoding="utf-8-sig")
    print(f"\n💾 [Saved CSV] Successfully updated {len(df_results)} questions to '{csv_save_path}'")
    
    # 2) TXT 파일 갱신 저장 (질문만 엔터 구분으로 저장)
    txt_save_path = PROJECT_ROOT / "eval" / "CATIA_RAG_High_Gap_Questions_Only.txt"
    with open(txt_save_path, "w", encoding="utf-8") as f_txt:
        for q_item in df_results["question"].dropna():
            f_txt.write(str(q_item).strip() + "\n\n")
    print(f"💾 [Saved TXT] Successfully updated question text list to '{txt_save_path}'")
else:
    print("⚠️ 수집된 고격차 질문이 없습니다.")


    Qwen 2.5 0.5B Upstage Solar Auto-Generated High-Gap QA Performance Summary         


,Model,Target High-Gap Questions,Direct LLM Solar Avg (1~5점),RAG LLM Solar Avg (1~5점),Solar Score Improvement (+Δ점수),Direct LLM BERT F1,RAG LLM BERT F1,BERT F1 Improvement (+ΔF1)
0,Qwen 2.5 0.5B,18,1.56 / 5.0,2.89 / 5.0,+1.33점,0.6179,0.6508,+0.0329



🔹 [선별된 RAG 고격차 질문 세부 점수 표]


,id,question,solar_direct_score,solar_rag_score,solar_score_gap,direct_f1,rag_f1,bert_f1_gap
0,Q01,CATIA V5에서 복사된 요소를 시스템 클립보드에 저장하는 기능은 어떤 툴 아이콘...,2,3,1,0.5338,0.6733,0.1395
1,Q02,CATIA V5에서 Parameterization Analysis 도구를 사용하여 ...,1,2,1,0.7112,0.7130,0.0019
2,Q03,CATIA V5에서 사용자 정의 뷰를 생성하고 저장된 뷰로 즉시 전환하는 정확한 메...,2,3,1,0.7040,0.6693,-0.0346
3,Q04,CATIA V5에서 'Reference Elements (Extended)' 기능으...,1,2,1,0.7324,0.6331,-0.0993
4,Q05,CATIA V5에서 Specification Tree를 실시간으로 숨기거나 표시하는...,2,5,3,0.5275,0.6033,0.0757
5,Q06,CATIA V5에서 'Part Design Workbench'를 즉시 활성화하기 위...,2,5,3,0.5437,0.6144,0.0707
6,Q07,CATIA V5에서 3D 제약 조건(Constraint)이 참조 차원(Dimensi...,1,2,1,0.5867,0.5934,0.0067
7,Q08,CATIA V5의 Hole 작업에서 'V-Bottom Type'을 지정하기 위해 대...,2,3,1,0.6729,0.6670,-0.0059
8,Q09,CATIA V5에서 파트 디자인과 어셈블리 워크벤치에 공통으로 사용되지만 기본적으로...,1,2,1,0.6331,0.6661,0.0331
9,Q10,CATIA V5에서 'Reference Elements (Extended)' 기능 ...,1,2,1,0.6351,0.6383,0.0032



💾 [Saved] Successfully exported 18 high-gap questions to 'C:\KDT_14\[11]Transformer\project\team-03-project\eval\CATIA_RAG_High_Gap_Auto_Questions.csv'
